# PR5: Data Preprocessing and EDA (Titanic dataset)

**Aim:** Clean the data, fill missing values, scale features, encode text columns and do basic statistics.

Steps in order: **Load -> Clean -> Missing values -> Encode -> Scale -> Statistics**

## 1. Load the dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

print(df.shape)
df.head()

In [ ]:
df.info()

## 2. Data cleaning

- Remove duplicate rows.
- Drop columns that are not useful: `PassengerId`, `Name`, `Ticket`, `Cabin` (Cabin has too many missing values).

In [ ]:
print("Duplicates:", df.duplicated().sum())
df = df.drop_duplicates()

df = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])
df.head()

## 3. Missing value handling

In [ ]:
print(df.isnull().sum())

- `Age` is a number, so fill with the **median**.
- `Embarked` is text, so fill with the **mode** (most common value).

In [ ]:
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

print(df.isnull().sum())

## 4. Encoding (text to numbers)

- `Sex` has 2 values, so **label encoding** (male=0, female=1).
- `Embarked` has 3 values, so **one-hot encoding** (`get_dummies`).

In [ ]:
df["Sex"] = df["Sex"].map({"male": 0, "female": 1})

df = pd.get_dummies(df, columns=["Embarked"], dtype=int)
df.head()

## 5. Feature scaling

- **Standardization:** (x - mean) / std. Result has mean 0 and std 1.
- **Min-Max scaling:** (x - min) / (max - min). Result is between 0 and 1.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

cols = ["Age", "Fare"]

df_std = df.copy()
df_std[cols] = StandardScaler().fit_transform(df[cols])

df_mm = df.copy()
df_mm[cols] = MinMaxScaler().fit_transform(df[cols])

print("Standardized:")
print(df_std[cols].describe().loc[["mean", "std", "min", "max"]])
print()
print("Min-Max:")
print(df_mm[cols].describe().loc[["mean", "std", "min", "max"]])

## 6. Statistical analysis

In [ ]:
print(df.describe())

In [ ]:
print("Mean Age   :", df["Age"].mean())
print("Median Age :", df["Age"].median())
print("Mode Age   :", df["Age"].mode()[0])
print("Std of Fare:", df["Fare"].std())
print("Skewness of Fare:", df["Fare"].skew())

In [ ]:
# Survival rate by group
print(df.groupby("Sex")["Survived"].mean())
print()
print(df.groupby("Pclass")["Survived"].mean())

Correlation: values near +1 or -1 mean a strong relationship.

In [ ]:
corr = df.corr()
print(corr["Survived"].sort_values())

In [ ]:
df["Age"].plot(kind="hist", bins=20, title="Age distribution")
plt.show()

df.groupby("Sex")["Survived"].mean().plot(kind="bar", title="Survival rate by Sex (0=male, 1=female)")
plt.show()

## Viva points

- **Median** is used for numeric gaps because it is not affected by outliers. **Mode** is used for text columns.
- **Label encoding** gives numbers 0,1,2... (fine for 2 categories). **One-hot** makes a separate 0/1 column per category (no false order).
- **Standardization** vs **Min-Max**: standardization is better when data has outliers.
- Scaling is needed for distance based models (KNN, SVM) and gradient based models.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(URL)
print("Shape:", df.shape)
print("Missing values:\n", df.isnull().sum())

# 1. Data cleaning
df = df.drop_duplicates()
df = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])  # irrelevant / ~77% missing

# 2. Missing value handling
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
print("\nMissing after handling:", df.isnull().sum().sum())

# 3. Statistical analysis
print("\n", df.describe())
print("\nCorrelation with Survived:\n", df.corr(numeric_only=True)["Survived"])

# 4. Visualization (histograms and boxplots)
num = ["Age", "Fare", "SibSp", "Parch"]
df[num].hist(bins=20, figsize=(10, 6))
plt.tight_layout()
plt.show()
df[num].plot(kind="box", subplots=True, layout=(1, 4), figsize=(12, 4))
plt.tight_layout()
plt.show()

# 5. Encoding
df["Sex"] = df["Sex"].map({"male": 0, "female": 1})            # label encoding
df = pd.get_dummies(df, columns=["Embarked"], drop_first=True)  # one-hot encoding

# 6. Train-test split (before scaling to avoid data leakage)
X, y = df.drop(columns="Survived"), df["Survived"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 7. Feature scaling (fit on train only)
cols = ["Age", "Fare"]
scaler = StandardScaler()
X_train, X_test = X_train.copy(), X_test.copy()
X_train[cols] = scaler.fit_transform(X_train[cols])
X_test[cols] = scaler.transform(X_test[cols])

print("\nTrain:", X_train.shape, "Test:", X_test.shape)
print(X_train.head())